# CropShield AI — CycleGAN Leaf-Lesion Training

This notebook trains the CycleGAN used by the CropShield AI Flask app to translate **healthy leaves ↔ diseased leaves**.

**Run on a GPU runtime:** `Runtime → Change runtime type → T4 GPU`.

Pipeline:

1. Install PyTorch (Colab already has it) and unpack the project.
2. Point `DATASET_DIR` at class-folder data (your own upload, Google Drive, or PlantVillage).
3. `gan.prepare_dataset` builds `trainA/trainB/testA/testB` for maize and wheat.
4. `gan.train_cyclegan` trains one compact ResNet CycleGAN per crop.
5. Download `checkpoints.zip` and copy its folders into `model/gan/` in the project.

Expected class-folder names must contain the crop and `healthy`, for example:

- `Corn_(maize)___healthy`, `Corn_(maize)___Common_rust_`
- `Wheat___healthy`, `Wheat___Yellow_rust`, `Wheat___Brown_rust`

or any `maize/` and `wheat/` folders containing class subfolders.

## 1. Check the GPU and install dependencies

In [ ]:
!nvidia-smi

import torch, torchvision
print('torch', torch.__version__, 'torchvision', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime -> Change runtime type -> T4 GPU'

!pip -q install pillow tqdm

## 2. Provide the project code

Upload `CropShield_AI_Project.zip`, mount Google Drive, or clone your Git repo.
Change `PROJECT_DIR` if the extracted folder has a different name.

In [ ]:
import os, zipfile, shutil

PROJECT_DIR = '/content/Crop_Disease_Enhanced'

# Option A: upload the project ZIP from your computer
if not os.path.isdir(PROJECT_DIR):
    from google.colab import files
    uploaded = files.upload()  # choose CropShield_AI_Project.zip
    for name in uploaded:
        with zipfile.ZipFile(name) as z:
            z.extractall('/content')

# Option B: mount Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copytree('/content/drive/MyDrive/CropShield/Crop_Disease_Enhanced', PROJECT_DIR)

assert os.path.isdir(PROJECT_DIR), f'Project not found at {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
print(os.listdir('.')[:15])

## 3. Provide the dataset

Set `DATASET_DIR` to a folder of class subfolders.

**Option A — upload your own ZIP:** run the next cell with `USE_UPLOAD = True`.

**Option B — Google Drive:** set `DATASET_DIR` to your Drive path.

**Option C — PlantVillage from Kaggle:** needs a Kaggle API token (`kaggle.json`).

In [ ]:
USE_UPLOAD = False
DATASET_DIR = '/content/dataset'   # where the class folders will live

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()  # choose your dataset ZIP
    os.makedirs(DATASET_DIR, exist_ok=True)
    for name in uploaded:
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(name) as z:
                z.extractall(DATASET_DIR)

# Option C: Kaggle PlantVillage (uncomment and upload kaggle.json first)
# os.makedirs('/root/.kaggle', exist_ok=True)
# shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
# !chmod 600 /root/.kaggle/kaggle.json
# !kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content --unzip
# DATASET_DIR = '/content/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train'

print('DATASET_DIR =', DATASET_DIR)
print('exists:', os.path.isdir(DATASET_DIR))
if os.path.isdir(DATASET_DIR):
    print(sorted(os.listdir(DATASET_DIR))[:10])

## 4. Build the unpaired CycleGAN dataset

This creates `data/cyclegan/maize/trainA` (healthy), `trainB` (diseased), plus test splits.

In [ ]:
!python -m gan.prepare_dataset --source "$DATASET_DIR" --output data/cyclegan --crop all --test-split 0.15

!find data/cyclegan -maxdepth 2 -type d | sort
!for d in data/cyclegan/*/train*; do echo -n "$d: "; ls "$d" | wc -l; done

## 5. Train the maize CycleGAN

Defaults: 128×128, batch size 1, 50 epochs. On a T4 this is roughly 35–60 minutes for a few hundred images.

Reduce `EPOCHS` to 20–30 for a quick demo, or raise `IMG_SIZE` to 256 only if you have plenty of GPU time.

In [ ]:
EPOCHS = 50
IMG_SIZE = 128

!python -m gan.train_cyclegan \
  --dataroot data/cyclegan/maize \
  --name maize \
  --img-size $IMG_SIZE \
  --epochs $EPOCHS \
  --batch-size 1 \
  --sample-every 5

## 6. Train the wheat CycleGAN (optional)

Skip this cell if you only have maize data.

In [ ]:
import os
if os.path.isdir('data/cyclegan/wheat/trainA') and os.path.isdir('data/cyclegan/wheat/trainB'):
    !python -m gan.train_cyclegan \
      --dataroot data/cyclegan/wheat \
      --name wheat \
      --img-size $IMG_SIZE \
      --epochs $EPOCHS \
      --batch-size 1 \
      --sample-every 5
else:
    print('No wheat dataset found, skipping.')

## 7. Inspect the generated previews

Each row is `real A | G(A) | F(G(A))` and `real B | G(B) | F(G(B))`.

In [ ]:
from IPython.display import Image, display
import glob

for crop in ['maize', 'wheat']:
    samples = sorted(glob.glob(f'model/gan/{crop}/samples/*.png'))
    if samples:
        print(crop, '->', samples[-1])
        display(Image(samples[-1], width=640))

## 8. Quick inference test

In [ ]:
import glob, os
test_images = glob.glob('data/cyclegan/maize/testA/*')[:1]
if test_images and os.path.exists('model/gan/maize/G_AB.pth'):
    !python -m gan.translate --checkpoint model/gan/maize --input "{test_images[0]}" --output /content/lesion_preview.jpg
    display(Image('/content/lesion_preview.jpg', width=320))
else:
    print('No maize test image or checkpoint.')

## 9. Download the checkpoints

Extract `checkpoints.zip` into your project so the layout is:

```
model/gan/maize/G_AB.pth
model/gan/maize/G_BA.pth
model/gan/maize/D_A.pth
model/gan/maize/D_B.pth
model/gan/maize/config.json
model/gan/wheat/...
```

Only `G_AB.pth` and `config.json` are required by the Flask app; the rest are kept for resuming training.

In [ ]:
!cd model/gan && zip -r /content/checkpoints.zip maize wheat -x '*/samples/*' || true
!ls -lh /content/checkpoints.zip

from google.colab import files
files.download('/content/checkpoints.zip')